In [25]:
# Import all needed packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import itertools
from xgboost import XGBClassifier, XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV

In [3]:
# Load CSV
df = pd.read_csv("fixed_final_data_product.csv", low_memory=False)
df

,lap_id,invalid_lap,BPS_SPEED,BPS_THROTTLE,BPS_STEER,BPS_BRAKE,BPS_CURRENTLAPTIMEINMS,BPS_LAPDISTANCE,BPS_WORLDPOSITIONX,BPS_WORLDPOSITIONY,...,dist_530_YAW,dist_530_PITCH,dist_530_ROLL,dist_530_left_dist,dist_530_right_dist,dist_530_dist_apex_1,dist_530_dist_apex_2,dist_530_angle_to_apex1,dist_530_angle_to_apex2,dist_530_proj_from_ref
0,10021698834789871149_1,1,316.0,1.000000,0.074568,0.0,2615.0,223.0,274.071987,313.458779,...,-2.715189,0.000475,-0.016953,1.741342,10.851370,132.666175,33.599712,-177.333510,-157.363838,3.100772
1,10021698834789871149_10,1,324.0,0.970802,0.000464,0.0,3097.0,273.0,310.096901,278.846172,...,-2.677639,-0.002429,-0.010994,5.229576,7.362893,134.315712,34.235318,-178.276773,-162.785343,0.365266
2,10021698834789871149_11,1,322.0,1.000000,-0.002375,0.0,2976.0,258.0,298.743940,288.688005,...,-2.672484,-0.003209,-0.015086,4.491344,8.101110,133.962370,34.073817,-177.906287,-161.483133,0.368740
3,10021698834789871149_12,0,322.0,0.025177,0.006011,0.0,3049.0,269.0,307.165908,281.569461,...,-2.683784,-0.003549,-0.013458,4.000375,8.592165,133.726732,33.972230,-177.665140,-160.617290,0.856654
4,10021698834789871149_13,1,324.0,1.000000,0.021114,0.0,3046.0,268.0,306.397540,282.215448,...,-2.576070,0.002121,-0.016932,5.969796,6.622633,134.674490,34.413890,-179.086529,-164.518584,1.101108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1027,9874269645241895165_2,1,144.0,0.000000,-0.525250,0.0,6234.0,450.0,369.369267,134.915112,...,-2.670881,0.003020,-0.000062,0.781471,11.886334,128.781581,29.677781,-178.004607,-156.571125,3.664728
1028,9941972541231747669_1,1,324.0,1.000000,0.000000,0.0,2979.0,263.0,302.615240,285.497492,...,-2.689289,-0.003212,-0.016119,2.482649,10.109993,133.011217,33.706067,-177.128822,-158.125436,2.364715
1029,9941972541231747669_2,0,324.0,1.000000,-0.000000,0.0,3014.0,264.0,303.184481,284.646456,...,-2.691542,-0.003572,-0.011709,3.187732,9.404851,133.342247,33.821835,-177.350536,-159.259773,1.664493
1030,9948558370850722411_1,1,324.0,1.000000,-0.003344,0.0,3149.0,276.0,301.987594,286.287858,...,-2.532977,0.004081,-0.015205,1.779133,10.876460,130.089505,30.731174,-178.246422,-158.331977,2.772034


In [7]:
target_col = "Target_CURRENTLAPTIMEINMS"
prefixes = ['BPS', 'STS', 'BPE', 'STM', 'STE', 'THE', 'THS']

# Keep only driver-controllable variables 
keep_cols = ['lap_id', 'BPS_SPEED', 'BPS_THROTTLE', 'BPS_STEER', 'BPS_BRAKE', 'BPS_CURRENTLAPTIMEINMS', 
             'BPS_LAPDISTANCE', 'BPS_WORLDPOSITIONX', 'BPS_WORLDPOSITIONY', 'BPE_SPEED', 'BPE_THROTTLE', 
             'BPE_STEER', 'BPE_BRAKE', 'BPE_CURRENTLAPTIMEINMS', 'BPE_LAPDISTANCE', 'BPE_WORLDPOSITIONX', 'BPE_WORLDPOSITIONY',
            'THS_SPEED', 'THS_THROTTLE', 'THS_STEER', 'THS_BRAKE', 'THS_CURRENTLAPTIMEINMS', 'THS_LAPDISTANCE', 'THS_WORLDPOSITIONX', 
             'THS_WORLDPOSITIONY', 'THE_SPEED', 'THE_THROTTLE', 'THE_STEER', 'THE_BRAKE', 'THE_CURRENTLAPTIMEINMS', 
             'THE_LAPDISTANCE', 'THE_WORLDPOSITIONX', 'THE_WORLDPOSITIONY', 'STS_SPEED', 'STS_THROTTLE', 'STS_STEER', 'STS_BRAKE', 
             'STS_CURRENTLAPTIMEINMS', 'STS_LAPDISTANCE', 'STS_WORLDPOSITIONX', 'STS_WORLDPOSITIONY', 'STM_SPEED', 'STM_THROTTLE', 
             'STM_STEER', 'STM_BRAKE', 'STM_CURRENTLAPTIMEINMS', 'STM_LAPDISTANCE', 'STM_WORLDPOSITIONX', 'STM_WORLDPOSITIONY', 
            'STE_SPEED', 'STE_THROTTLE', 'STE_STEER', 'STE_BRAKE', 'STE_CURRENTLAPTIMEINMS', 'STE_LAPDISTANCE', 'STE_WORLDPOSITIONX', 
             'STE_WORLDPOSITIONY', 'Target_CURRENTLAPTIMEINMS']

In [12]:
# Helper functions 
def apply_iqr_filter(data, target_col, prefixes):
    """Apply IQR filtering to target and selected prefixes."""
    data = data.copy()
    # Filter target
    Q1 = data[target_col].quantile(0.25)
    Q3 = data[target_col].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    data = data[data[target_col] <= upper_bound]

    # Filter chosen prefixes
    for p in prefixes:
        col = f"{p}_CURRENTLAPTIMEINMS"
        if col in data.columns:
            Q1 = data[col].quantile(0.25)
            Q3 = data[col].quantile(0.75)
            IQR = Q3 - Q1
            upper_bound = Q3 + 1.5 * IQR
            data = data[data[col] <= upper_bound]
    return data

def adjusted_r2(y_true, y_pred, n_features):
    """Compute adjusted R²."""
    r2 = r2_score(y_true, y_pred)
    n = len(y_true)
    p = n_features
    return 1 - (1 - r2) * (n - 1) / (n - p - 1)

def evaluate_model(df, target_col):
    """Train/test split and evaluate model."""
    df = df.dropna()
    df2 = df.drop(columns=['lap_id'])
    X = df2.drop(columns=[target_col])
    y = df2[target_col]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    model = XGBRegressor(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    r2_adj = adjusted_r2(y_test, y_pred, X_test.shape[1])
    return mse, rmse, mae, r2, r2_adj

In [45]:
results = []

for r in range(len(prefixes) + 1):
    for combo in itertools.combinations(prefixes, r):
        filtered_df = apply_iqr_filter(df[keep_cols], target_col, combo)
        mse, rmse, mae, r2, adj_r2 = evaluate_model(filtered_df, target_col)
        results.append((combo, mse, rmse, mae, r2, adj_r2))
        print(f"Tested {combo or 'no prefixes'} → MSE={mse:.2f}, RMSE={rmse:.2f}, MAE={mae:.2f}, R²={r2:.4f}, A-R²={adj_r2:.4f}")

# Find best combination
results_df = pd.DataFrame(results, columns=["prefixes", "MSE", "RMSE", "MAE", "R2", "ADJ_R2"])
best_row = results_df.loc[results_df["MSE"].idxmin()]

print("\n=== Best Subset Result ===")
print(f"Best prefixes: {best_row['prefixes'] or 'no prefixes'}")
print(f"Lowest MSE: {best_row['MSE']:.2f}")
print(f"RMSE: {best_row['RMSE']:.2f}")
print(f"MAE: {best_row['MAE']:.2f}")
print(f"R²: {best_row['R2']:.4f}")
print(f"Adjusted R²: {best_row['ADJ_R2']:.4f}")

Tested no prefixes → MSE=110225.02, RMSE=332.00, MAE=189.15, R²=0.8767, A-R²=0.7992
Tested ('BPS',) → MSE=49358.95, RMSE=222.17, MAE=136.18, R²=0.8973, A-R²=0.8245
Tested ('STS',) → MSE=71293.59, RMSE=267.01, MAE=175.58, R²=0.9111, A-R²=0.8455
Tested ('BPE',) → MSE=104665.96, RMSE=323.52, MAE=198.85, R²=0.8771, A-R²=0.7961
Tested ('STM',) → MSE=63891.48, RMSE=252.77, MAE=156.95, R²=0.8981, A-R²=0.8209
Tested ('STE',) → MSE=102674.47, RMSE=320.43, MAE=187.19, R²=0.8986, A-R²=0.8310
Tested ('THE',) → MSE=131623.52, RMSE=362.80, MAE=186.87, R²=0.8429, A-R²=0.7370
Tested ('THS',) → MSE=135764.42, RMSE=368.46, MAE=211.74, R²=0.8411, A-R²=0.7312
Tested ('BPS', 'STS') → MSE=81100.94, RMSE=284.78, MAE=173.08, R²=0.8691, A-R²=0.7613
Tested ('BPS', 'BPE') → MSE=89504.99, RMSE=299.17, MAE=162.50, R²=0.8864, A-R²=0.8049
Tested ('BPS', 'STM') → MSE=74021.03, RMSE=272.07, MAE=151.80, R²=0.8159, A-R²=0.6597
Tested ('BPS', 'STE') → MSE=139612.09, RMSE=373.65, MAE=195.31, R²=0.8171, A-R²=0.6805
Tested 

In [18]:
df = pd.read_csv("fixed_final_data_product.csv", low_memory=False)
df

,lap_id,invalid_lap,BPS_SPEED,BPS_THROTTLE,BPS_STEER,BPS_BRAKE,BPS_CURRENTLAPTIMEINMS,BPS_LAPDISTANCE,BPS_WORLDPOSITIONX,BPS_WORLDPOSITIONY,...,dist_530_YAW,dist_530_PITCH,dist_530_ROLL,dist_530_left_dist,dist_530_right_dist,dist_530_dist_apex_1,dist_530_dist_apex_2,dist_530_angle_to_apex1,dist_530_angle_to_apex2,dist_530_proj_from_ref
0,10021698834789871149_1,1,316.0,1.000000,0.074568,0.0,2615.0,223.0,274.071987,313.458779,...,-2.715189,0.000475,-0.016953,1.741342,10.851370,132.666175,33.599712,-177.333510,-157.363838,3.100772
1,10021698834789871149_10,1,324.0,0.970802,0.000464,0.0,3097.0,273.0,310.096901,278.846172,...,-2.677639,-0.002429,-0.010994,5.229576,7.362893,134.315712,34.235318,-178.276773,-162.785343,0.365266
2,10021698834789871149_11,1,322.0,1.000000,-0.002375,0.0,2976.0,258.0,298.743940,288.688005,...,-2.672484,-0.003209,-0.015086,4.491344,8.101110,133.962370,34.073817,-177.906287,-161.483133,0.368740
3,10021698834789871149_12,0,322.0,0.025177,0.006011,0.0,3049.0,269.0,307.165908,281.569461,...,-2.683784,-0.003549,-0.013458,4.000375,8.592165,133.726732,33.972230,-177.665140,-160.617290,0.856654
4,10021698834789871149_13,1,324.0,1.000000,0.021114,0.0,3046.0,268.0,306.397540,282.215448,...,-2.576070,0.002121,-0.016932,5.969796,6.622633,134.674490,34.413890,-179.086529,-164.518584,1.101108
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1027,9874269645241895165_2,1,144.0,0.000000,-0.525250,0.0,6234.0,450.0,369.369267,134.915112,...,-2.670881,0.003020,-0.000062,0.781471,11.886334,128.781581,29.677781,-178.004607,-156.571125,3.664728
1028,9941972541231747669_1,1,324.0,1.000000,0.000000,0.0,2979.0,263.0,302.615240,285.497492,...,-2.689289,-0.003212,-0.016119,2.482649,10.109993,133.011217,33.706067,-177.128822,-158.125436,2.364715
1029,9941972541231747669_2,0,324.0,1.000000,-0.000000,0.0,3014.0,264.0,303.184481,284.646456,...,-2.691542,-0.003572,-0.011709,3.187732,9.404851,133.342247,33.821835,-177.350536,-159.259773,1.664493
1030,9948558370850722411_1,1,324.0,1.000000,-0.003344,0.0,3149.0,276.0,301.987594,286.287858,...,-2.532977,0.004081,-0.015205,1.779133,10.876460,130.089505,30.731174,-178.246422,-158.331977,2.772034


In [ ]:
# Drop the best prefixes
best_prefixes = ['BPS', 'BPE', 'STE', 'THS']  # from best combo result
target_col = "Target_CURRENTLAPTIMEINMS"

# Keep only driver-controllable variables 
keep_cols = ['lap_id', 'BPS_SPEED', 'BPS_THROTTLE', 'BPS_STEER', 'BPS_BRAKE', 'BPS_CURRENTLAPTIMEINMS', 
             'BPS_LAPDISTANCE', 'BPS_WORLDPOSITIONX', 'BPS_WORLDPOSITIONY', 'BPE_SPEED', 'BPE_THROTTLE', 
             'BPE_STEER', 'BPE_BRAKE', 'BPE_CURRENTLAPTIMEINMS', 'BPE_LAPDISTANCE', 'BPE_WORLDPOSITIONX', 'BPE_WORLDPOSITIONY',
            'THS_SPEED', 'THS_THROTTLE', 'THS_STEER', 'THS_BRAKE', 'THS_CURRENTLAPTIMEINMS', 'THS_LAPDISTANCE', 'THS_WORLDPOSITIONX', 
             'THS_WORLDPOSITIONY', 'THE_SPEED', 'THE_THROTTLE', 'THE_STEER', 'THE_BRAKE', 'THE_CURRENTLAPTIMEINMS', 
             'THE_LAPDISTANCE', 'THE_WORLDPOSITIONX', 'THE_WORLDPOSITIONY', 'STS_SPEED', 'STS_THROTTLE', 'STS_STEER', 'STS_BRAKE', 
             'STS_CURRENTLAPTIMEINMS', 'STS_LAPDISTANCE', 'STS_WORLDPOSITIONX', 'STS_WORLDPOSITIONY', 'STM_SPEED', 'STM_THROTTLE', 
             'STM_STEER', 'STM_BRAKE', 'STM_CURRENTLAPTIMEINMS', 'STM_LAPDISTANCE', 'STM_WORLDPOSITIONX', 'STM_WORLDPOSITIONY', 
            'STE_SPEED', 'STE_THROTTLE', 'STE_STEER', 'STE_BRAKE', 'STE_CURRENTLAPTIMEINMS', 'STE_LAPDISTANCE', 'STE_WORLDPOSITIONX', 
             'STE_WORLDPOSITIONY', 'Target_CURRENTLAPTIMEINMS']


In [22]:
df1 = df[keep_cols]

In [23]:
# Clean based on best prefixes 
Q1 = df1[target_col].quantile(0.25)
Q3 = df1[target_col].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
df1 = df1[df1[target_col] <= upper_bound]

# Filter chosen prefixes
for p in best_prefixes:
    col = f"{p}_CURRENTLAPTIMEINMS"
    if col in df1.columns:
        Q1 = df1[col].quantile(0.25)
        Q3 = df1[col].quantile(0.75)
        IQR = Q3 - Q1
        upper_bound = Q3 + 1.5 * IQR
        df1 = df1[df1[col] <= upper_bound]

In [24]:
X = df1.drop(columns=['lap_id', target_col])
y = df1[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
r2_adj = adjusted_r2(y_test, y_pred, X_test.shape[1])

print(f"R²:   {r2:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"Adjusted R²: {r2_adj:.4f}")

R²:   0.9161
MSE:  37816.4414
RMSE: 194.4645
MAE:  129.6379
Adjusted R²: 0.8490


In [26]:
df1

,lap_id,BPS_SPEED,BPS_THROTTLE,BPS_STEER,BPS_BRAKE,BPS_CURRENTLAPTIMEINMS,BPS_LAPDISTANCE,BPS_WORLDPOSITIONX,BPS_WORLDPOSITIONY,BPE_SPEED,...,STM_WORLDPOSITIONY,STE_SPEED,STE_THROTTLE,STE_STEER,STE_BRAKE,STE_CURRENTLAPTIMEINMS,STE_LAPDISTANCE,STE_WORLDPOSITIONX,STE_WORLDPOSITIONY,Target_CURRENTLAPTIMEINMS
0,10021698834789871149_1,316.0,1.000000,0.074568,0.0,2615.0,223.0,274.071987,313.458779,135.0,...,148.202863,133.364860,1.000000,4.232725e-16,0.0,9124.121494,508.364860,374.206900,78.223315,15377
1,10021698834789871149_10,324.0,0.970802,0.000464,0.0,3097.0,273.0,310.096901,278.846172,240.0,...,154.219878,235.000000,1.000000,2.389582e-16,0.0,7965.937677,581.395845,409.517581,15.863695,12146
2,10021698834789871149_11,322.0,1.000000,-0.002375,0.0,2976.0,258.0,298.743940,288.688005,254.0,...,159.573540,248.000000,1.000000,3.807631e-04,0.0,8448.000000,606.000000,427.998102,-0.447750,12220
3,10021698834789871149_12,322.0,0.025177,0.006011,0.0,3049.0,269.0,307.165908,281.569461,291.0,...,154.030789,239.867059,1.000000,3.621235e-17,0.0,8216.005891,592.867059,417.804259,7.805158,12219
4,10021698834789871149_13,324.0,1.000000,0.021114,0.0,3046.0,268.0,306.397540,282.215448,210.0,...,140.429883,181.372113,1.000000,-5.117434e-17,0.0,7680.070146,537.372113,382.093288,50.626736,12881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1025,9867356207088057695_5,325.0,0.959060,-0.000485,0.0,2639.0,232.0,279.897861,306.563766,212.0,...,161.603898,245.000000,1.000000,-7.632783e-17,0.0,8588.594248,604.828161,427.459209,0.714011,12395
1026,9874269645241895165_1,319.0,1.000000,0.004399,0.0,2580.0,222.0,269.904300,317.760615,158.0,...,163.951442,198.000000,1.000000,-1.955901e-16,0.0,8775.662860,580.203492,412.594393,21.076923,13303
1029,9941972541231747669_2,324.0,1.000000,-0.000000,0.0,3014.0,264.0,303.184481,284.646456,208.0,...,-7.262050,248.000000,1.000000,-1.443580e-11,0.0,8665.000000,616.000000,435.298159,-7.262050,12326
1030,9948558370850722411_1,324.0,1.000000,-0.003344,0.0,3149.0,276.0,301.987594,286.287858,164.0,...,166.822221,179.000000,0.631893,5.702903e-17,0.0,8408.278406,545.277811,380.480800,61.522218,13604


In [28]:
# Hyperparameter tuning 
param_dist = {
    'n_estimators': [250, 300, 350],
    'max_depth': [5, 6, 7],
    'learning_rate': [0.08, 0.1, 0.12],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    objective='reg:squarederror'
)

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=50,             
    scoring='neg_mean_squared_error',
    cv=5,                  
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

best_model = random_search.best_estimator_
print("Best parameters found:")
print(random_search.best_params_)

# Predictions
y_pred = best_model.predict(X_test)

# Metrics
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)
r2_adj = 1 - (1 - r2) * (len(y_test) - 1) / (len(y_test) - X_test.shape[1] - 1)

print(f"\nBest Model Performance:")
print(f"R²:   {r2:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"Adjusted R²: {r2_adj:.4f}")

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best parameters found:
{'subsample': 0.9, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.12, 'colsample_bytree': 0.8}

Best Model Performance:
R²:   0.9202
MSE:  35975.9141
RMSE: 189.6732
MAE:  127.3095
Adjusted R²: 0.8564
